# The Markov blanket

_Artificial Intelligence — more_

**A node only needs to know its neighbors. Given them, the rest of the world is irrelevant.**

In a Bayes net, a node is not tangled up with the whole graph. It depends on a small local set: its Markov blanket.

     The blanket is the node's parents, its children, and its children's other parents (co-parents).

---

This notebook is a practice scaffold for the **The Markov blanket** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

In [ ]:
# Bayes net edges (parent -> child). Target node = "X".
parents = {"X": ["A", "B"], "Y": ["X", "C"], "Z": ["X", "D"], "W": ["E"]}

def children(node):
    return [n for n, ps in parents.items() if node in ps]

def markov_blanket(node):
    blanket = set(parents.get(node, []))          # parents
    for ch in children(node):                     # children ...
        blanket.add(ch)
        blanket.update(parents[ch])               # ... and their other parents (co-parents)
    blanket.discard(node)
    return sorted(blanket)

mb = markov_blanket("X")
print("children of X:", children("X"))
print("Markov blanket of X:", mb)
# E and W are outside the blanket -> X is independent of them given the blanket
print("W in blanket?", "W" in mb)

## Visualize the data & results

_Burglar Alarm network: given an earthquake and both neighbors calling, what is the local posterior that the alarm is on?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Burglar Alarm net. Observed: no burglary, an earthquake, John & Mary both called.
# Markov blanket of Alarm = parents {Burglary, Earthquake} + children {John, Mary}.
P_A_BE = np.zeros((2, 2, 2))                  # P(Alarm | Burglary, Earthquake)
P_A_BE[0,0] = [0.999, 0.001]; P_A_BE[0,1] = [0.71, 0.29]
P_A_BE[1,0] = [0.06, 0.94];   P_A_BE[1,1] = [0.05, 0.95]
P_J_A = np.array([[0.95, 0.05], [0.10, 0.90]])  # P(JohnCalls | Alarm)
P_M_A = np.array([[0.99, 0.01], [0.30, 0.70]])  # P(MaryCalls | Alarm)

B, E, J, M = 0, 1, 1, 1                        # no burglary, quake, both call
score = np.array([P_A_BE[B,E,a]*P_J_A[a,J]*P_M_A[a,M] for a in range(2)])
post = score / score.sum()                    # [P(off), P(on)] -> on = 0.998

labels = ["Alarm = On", "Alarm = Off"]
vals = [post[1], post[0]]
fig, ax = plt.subplots()
bars = ax.bar(labels, vals, color=["#7ee787", "#ff7b72"])
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v, str(round(v,3)), ha="center", va="bottom")
ax.set_title("P(Alarm | its Markov blanket) in the Burglar Alarm net")
plt.show()